# Qwen2.5-VL-72B on H100 — full Glück Bible OCR

Runs the whole Glück 1694 Latvian Fraktur Bible through Qwen2.5-VL-72B (AWQ int4) via vLLM on a single H100 80GB, outputs a JSON with `max_verse` per page plus a confidence score and alternatives. Target: ~20 minutes of inference for ~1500 pages, leaving your allowance with headroom to re-run uncertain pages.

## Why Qwen2.5-VL-72B over Qwen3.6-27B or Qwen3-VL-32B

Qwen3.6-27B is 15 hours old and is positioned as a coding/agentic model; its vision benchmarks (CC-OCR 81.2, OCRBench 89.4) are essentially tied with the previous generation rather than a leap. For historical document OCR specifically, Qwen2.5-VL-72B is more purpose-fit: it was trained explicitly for document understanding, is heavily benchmarked on OCR tasks, has the best OCRBench score in the Qwen VLM family, and has mature vLLM support. At int4 it's ~40GB on VRAM — fits on your H100 with plenty of room for long visual context windows.

If you'd rather try Qwen3-VL-32B (~21GB, release notes claim improved ancient-script OCR), change `MODEL` in cell 2. The rest of the notebook is model-agnostic.

## Time budget — roughly

| Step | Time |
|------|------|
| Install vLLM + deps | ~5 min |
| Download Qwen2.5-VL-72B-AWQ from HF | ~5–15 min (~40GB) |
| Start vLLM server | ~2 min |
| Pass 1: all 1500 pages | ~20–30 min |
| Pass 2: re-check uncertain pages | ~2–5 min |
| **Total** | **~35–55 min** |

H100 80GB is more than enough; most of this can run unattended.

## Run order

Cells 1–4: setup (run once)  
Cell 5: start vLLM server (leave running in a terminal, or use `%%bash --bg`)  
Cells 6–9: test on a few pages, verify prompt works  
Cells 10–11: full-book pass  
Cells 12–13: QA against your ground truth, re-check uncertain pages  
Cell 14: write final `bible_mapping.py`

## 1. Environment

Run once on your H100 machine. vLLM brings its own CUDA deps; a fresh venv is cleanest.

In [1]:
# Shell cell — run once per machine, NOT per notebook session.
# Skip if you already have vllm installed.
#
# !python -m venv ~/vlm-venv && source ~/vlm-venv/bin/activate  # optional
!pip install -q 'vllm>=0.6.5' 'openai>=1.50' 'pillow' 'requests' 'tqdm'
!pip install "numpy<2"
!pip show vllm | head -2
!nvidia-smi --query-gpu=name,memory.total,memory.free --format=csv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pandas 2.1.4 requires numpy<2,>=1.26.0; python_version >= "3.12", but you have numpy 2.2.6 which is incompatible.
matplotlib 3.8.2 requires numpy<2,>=1.21, but you have numpy 2.2.6 which is incompatible.
scipy 1.11.4 requires numpy<1.28.0,>=1.21.6, but you have numpy 2.2.6 which is incompatible.
scikit-learn 1.3.2 requires numpy<2.0,>=1.17.3, but you have numpy 2.2.6 which is incompatible.
Name: vllm
Version: 0.19.1
ERROR: Pipe to stdout was broken
name, memory.total [MiB], memory.free [MiB]
NVIDIA H200, 143771 MiB, 143167 MiB


## 2. Config

In [11]:
from pathlib import Path

# --- Model ---
# Pre-quantized AWQ int4 of Qwen2.5-VL-72B-Instruct. ~40GB on disk / ~42GB VRAM.
# If HF rate-limits or the repo moves, alternatives:
#   'Qwen/Qwen2.5-VL-72B-Instruct'          (BF16, ~145GB, won't fit on H100)
#   'Qwen/Qwen2.5-VL-32B-Instruct-AWQ'      (~18GB, faster, slightly lower accuracy)
#   'Qwen/Qwen3-VL-32B-Instruct'            (newer, BF16 ~60GB, tight fit)
#H100 (2 (!) SWITCHES)
MODEL = 'Qwen/Qwen2.5-VL-72B-Instruct-AWQ'
#H200
MODEL = 'Qwen/Qwen2.5-VL-72B-Instruct-FP8'
MODEL = 'RedHatAI/Qwen2.5-VL-72B-Instruct-FP8-dynamic'
# --- Data ---
MIRROR_BASE = 'https://t.noit.pro/gluck_1694/raw'   # yields .../NNNNN.jpg
MS_ID = 'bsb10914821'                                # Glück 1694
FIRST_PAGE = 13                                       # mirror's first page
LAST_PAGE  = 2690                                    # ADJUST if you know the real last page

# Local working dir
WORK = Path('~/gluck_vlm').expanduser()
WORK.mkdir(exist_ok=True)
CACHE = WORK / 'cache'; CACHE.mkdir(exist_ok=True)       # downloaded images
RESULTS = WORK / 'results'; RESULTS.mkdir(exist_ok=True) # JSON per page

# vLLM server endpoint (cell 5 starts this)
VLLM_URL = 'http://localhost:8000/v1'

# Inference
MAX_VERSE = 180          # Psalm 119 has 176
#H100
BATCH = 8                # pages per concurrent request batch
#H200
BATCH = 16
BATCH = 32  # H200 can absorb this, you're at 37% KV cache
IMG_MAX_PIXELS = 1024*1024  # resize cap before sending; more=better OCR, slower

print(f'Model:   {MODEL}')
print(f'Mirror:  {MIRROR_BASE}')
print(f'Pages:   {FIRST_PAGE} .. {LAST_PAGE}  ({LAST_PAGE - FIRST_PAGE + 1} total)')
print(f'Workdir: {WORK}')

Model:   RedHatAI/Qwen2.5-VL-72B-Instruct-FP8-dynamic
Mirror:  https://t.noit.pro/gluck_1694/raw
Pages:   13 .. 2690  (2678 total)
Workdir: /teamspace/studios/this_studio/gluck_vlm


## 3. Ground-truth (for QA later)

Paste your existing mapping here so we can measure accuracy on known pages and flag disagreements. Fractional codes (`.1`, `.5`, `.75`, negatives) are normalized to their integer part.

## 4. Image fetch helper

Downloads from your mirror to local cache, with retries. Fallback to BSB IIIF if the mirror 404s.

In [12]:
import io
import time
import requests
from PIL import Image


def page_url(page: int) -> str:
    return f'{MIRROR_BASE}/{page:05d}.jpg'


def iiif_url(page: int, width: int = 1600) -> str:
    return (f'https://api.digitale-sammlungen.de/iiif/image/v2/'
            f'{MS_ID}_{page:05d}/full/{width},/0/default.jpg')


def fetch_image(page: int, prefer_mirror: bool = True) -> Image.Image:
    """Return PIL image. Caches to disk."""
    local = CACHE / f'{page:05d}.jpg'
    if local.exists():
        return Image.open(local).convert('RGB')
    urls = [page_url(page), iiif_url(page)] if prefer_mirror else [iiif_url(page), page_url(page)]
    last_err = None
    for url in urls:
        for attempt in range(3):
            try:
                r = requests.get(url, timeout=30)
                if r.status_code == 200 and len(r.content) > 1000:
                    local.write_bytes(r.content)
                    return Image.open(local).convert('RGB')
                last_err = f'{url} -> HTTP {r.status_code}, {len(r.content)} bytes'
            except Exception as e:
                last_err = f'{url} -> {e}'
            time.sleep(1.5 * (attempt + 1))
    raise RuntimeError(f'Failed to fetch page {page}: {last_err}')


def shrink_for_vlm(img: Image.Image, max_pixels: int = IMG_MAX_PIXELS) -> Image.Image:
    """Scale to <= max_pixels (area). Preserves aspect ratio."""
    w, h = img.size
    scale = (max_pixels / (w * h)) ** 0.5
    if scale >= 1.0:
        return img
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

def shrink_for_vlm(img: Image.Image) -> Image.Image:
    """Target ~1600x2100 for 2-column Fraktur pages — enough pixels for
    reliable digit OCR without bloating the token budget."""
    TARGET_LONG_EDGE = 2100
    w, h = img.size
    long_edge = max(w, h)
    if long_edge <= TARGET_LONG_EDGE:
        return img
    scale = TARGET_LONG_EDGE / long_edge
    return img.resize((int(w * scale), int(h * scale)), Image.LANCZOS)

def body_crop(img: Image.Image) -> Image.Image:
    """Crop to body region, excluding marginalia columns and page headers/footers.
    Fractions were measured from pages 13/15/42 of this edition."""
    w, h = img.size
    box = (int(w * 0.13), int(h * 0.055), int(w * 0.84), int(h * 0.965))
    return img.crop(box)


# Quick test
img13 = fetch_image(13)
print(f'page 13: {img13.size}')
cropped = body_crop(img13)
shrunk  = shrink_for_vlm(cropped)
print(f'  body crop: {cropped.size}')
print(f'  shrunk:    {shrunk.size}')

page 13: (2063, 2656)
  body crop: (1464, 2417)
  shrunk:    (1271, 2100)


## 5. Start vLLM server

This takes ~2 minutes on first run (loads the model into VRAM). **Run in a separate terminal**, or use the magic below to background it inside the notebook.

For H100:

```bash
vllm serve Qwen/Qwen2.5-VL-72B-Instruct-AWQ \
  --port 8000 \
  --quantization awq \
  --dtype float16 \
  --max-model-len 8192 \
  --gpu-memory-utilization 0.92
```

FOR H200: --max-model-len 16384
```bash
vllm serve RedHatAI/Qwen2.5-VL-72B-Instruct-FP8-dynamic \
  --port 8000 \
  --max-model-len 16384 \
  --gpu-memory-utilization 0.92 \
  --trust-remote-code
```
Flags explained:
- `--max-model-len 8192` — more than enough for a one-image, short-prompt task. Lower = more KV-cache headroom for batching.
- `--limit-mm-per-prompt image=1` — we send one image per request.
- `--gpu-memory-utilization 0.92` — H100 has 80GB, model takes ~42GB, this gives KV cache ~33GB → room for big batches.

Wait until the log shows `Uvicorn running on http://0.0.0.0:8000`. Then come back here.

In [13]:
# Sanity check: server is up
r = requests.get(f'{VLLM_URL}/models', timeout=5)
print(r.status_code, r.json() if r.status_code == 200 else r.text)

200 {'object': 'list', 'data': [{'id': 'RedHatAI/Qwen2.5-VL-72B-Instruct-FP8-dynamic', 'object': 'model', 'created': 1777012337, 'owned_by': 'vllm', 'root': 'RedHatAI/Qwen2.5-VL-72B-Instruct-FP8-dynamic', 'parent': None, 'max_model_len': 16384, 'permission': [{'id': 'modelperm-90c594984608dc45', 'object': 'model_permission', 'created': 1777012337, 'allow_create_engine': False, 'allow_sampling': True, 'allow_logprobs': True, 'allow_search_indices': False, 'allow_view': True, 'allow_fine_tuning': False, 'organization': '*', 'group': None, 'is_blocking': False}]}]}


## 6. Prompt

Names the three decoy sources (marginalia, summary block, header) explicitly. Asks for JSON with reading order, self-reported confidence, and digit-confusion alternatives.

In [14]:
PROMPT = """This image is one page of a two-column Bible printed in the \
Latvian language using Gothic Fraktur script (Glück 1694). Each body verse \
is introduced by an Arabic numeral followed by a period ("1.", "2.", "12.") \
at the start of its paragraph, in the main body text of either column.

TASK: recognize the text in this page, in reading order \
(left column top-to-bottom, then right column top-to-bottom). Ignore sidenotes. \
skip decorative images that sometimes appear. The language is old Latvian written \
with German alphabet, having extra inventions like n with macron, striped n and others. \
Therefore do not proof the text, just recognize. It also might have cross sometimes in text, \
recognize that as †.
Output each verse's text verbatim, preserving archaic spellings, foreign characters, punctuation, and line breaks and \\ marks.
If you cannot read a character, output U+FFFD (the Unicode replacement character) rather than guessing.
Do NOT normalize spelling. Do NOT substitute modern Latvian equivalents. 
The goal is faithful transcription of a 1694 print, not a readable modern text.
Reply with ONLY a JSON object, no prose, no markdown fences, exactly this shape:

{
  "verses": [
    {"n": 1, "column": "left", "text": ""},
    {"n": 12, "column": "right", "text": ""}
  ],
  "confidence": "high",
  "notes": ""
}

Field details:
  - "n": the verse number (integer, 1..180).
  - "column": "left" or "right".
  - "text": your recognized text in the respective column
  - "confidence": "high" / "medium" / "low".
  - "notes": brief free text only if something is wrong (damaged print, \
    missing verse number, etc). Empty string otherwise.

"""

print(f'prompt length: {len(PROMPT)} chars')

prompt length: 1646 chars


## 7. Single-page inference via OpenAI-compatible API

vLLM exposes `/v1/chat/completions`. The OpenAI client works directly.

In [9]:
import base64
import json
import re
from dataclasses import dataclass, field
from typing import Optional

from openai import OpenAI

_client = OpenAI(base_url=VLLM_URL, api_key='EMPTY')


def img_to_data_url(img: Image.Image, fmt: str = 'JPEG', quality: int = 90) -> str:
    buf = io.BytesIO()
    img.save(buf, format=fmt, quality=quality)
    return f'data:image/{fmt.lower()};base64,' + base64.b64encode(buf.getvalue()).decode('ascii')


def extract_json(text: str) -> dict:
    text = re.sub(r'^```(?:json)?\s*|\s*```\s*$', '', text.strip(), flags=re.M)
    m = re.search(r'\{.*\}', text, re.S)
    if not m:
        raise ValueError(f'no JSON in: {text[:200]!r}')
    return json.loads(m.group(0))


@dataclass
class PageResult:
    page: int
    max_verse: Optional[int] = None
    full_list: list = field(default_factory=list)
    alternatives: list = field(default_factory=list)
    chapter_starts_here: bool = False
    confidence: str = 'unknown'
    notes: str = ''
    raw: str = ''
    error: str = ''


def parse(page: int, raw: str) -> PageResult:
    try:
        data = extract_json(raw)
    except Exception as e:
        return PageResult(page=page, raw=raw, error=f'parse: {e}')
    verses = data.get('verses', [])
    nums = []
    alts = []
    for v in verses:
        try:
            n = int(v['n'])
        except (KeyError, ValueError, TypeError):
            continue
        if not (1 <= n <= MAX_VERSE):
            continue
        nums.append(n)
        a = [int(x) for x in v.get('alternatives', []) if str(x).isdigit()]
        a = [x for x in a if 1 <= x <= MAX_VERSE]
        if a:
            alts.append((n, a))
    return PageResult(
        page=page, raw=raw,
        max_verse=max(nums) if nums else None,
        full_list=nums, alternatives=alts,
        chapter_starts_here=bool(data.get('chapter_starts_here', False)),
        confidence=str(data.get('confidence', 'unknown')),
        notes=str(data.get('notes', '')),
    )


def extract_page(page: int, crop: bool = True) -> PageResult:
    cache_file = RESULTS / f'{page:05d}.json'
    if cache_file.exists():
        try:
            d = json.loads(cache_file.read_text())
            return PageResult(**d)
        except Exception:
            pass
    try:
        img = fetch_image(page)
        if crop:
            img = body_crop(img)
        img = shrink_for_vlm(img)
        resp = _client.chat.completions.create(
            model=MODEL, temperature=0.0, max_tokens=1024,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'image_url', 'image_url': {'url': img_to_data_url(img)}},
                    {'type': 'text', 'text': PROMPT},
                ],
            }],
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        raw = resp.choices[0].message.content
        result = parse(page, raw)
    except Exception as e:
        result = PageResult(page=page, error=str(e))
    # Persist
    cache_file.write_text(json.dumps(result.__dict__))
    return result


# Smoke test on three known pages
rl=[]
for p in [13, 15, 42]:
    r = extract_page(p)
    rl.append(r)
    # expected = PAGE_TO_EXPECTED.get(p, '?')
    # flag = 'OK' if r.max_verse == expected else 'MISMATCH'
    # print(f'p{p}: max={r.max_verse} (expected {expected}) {flag}  conf={r.confidence}  list={r.full_list}')
    # if r.error:
    #     print(f'  ERROR: {r.error}')
    # if r.notes:
    #     print(f'  notes: {r.notes}')

In [10]:
rl

[PageResult(page=13, max_verse=7, full_list=[1, 7], alternatives=[], chapter_starts_here=False, confidence='high', notes='', raw='{\n  "verses": [\n    {\n      "n": 1,\n      "column": "left",\n      "text": "Esahkumā rad-\\ndija Deews to\\nDebbesi un to\\nSemmi.\\n2. Kuta Sem-\\nme bija ne-istaifita un tuk-tcha /\\nun Tumsiba bija pahr teem Dsit-\\ntumeem / un tas Deewa Gars\\niddinajahs pa Uhdens Wirfit.\\n3. Un Deews fazzija: Lai\\nGaifma tohp: tad tappeGaifma.\\n4. Un Deews redseja/to Gaif-\\nnu labbu effam / tad schihre\\nDeews te Gaifmu no tahs Tuu-\\nibas.\\n5. Un Deews nosauze to Gaif-\\nnu Deena/ un to Tumsibu fan-\\ne winsch Nakts. Tad tappe no\\nWakkara un Rihta ta pirma\\nDeena.\\n6. Un Deews fazzija : Laid\\ntohp Isplattijums Uhdens star-\\npā/ un lat tas schihre Uhdeni no\\nUhdens."\n    },\n    {\n      "n": 7,\n      "column": "right",\n      "text": "Un Deews darrija to J\\nplattijumu un darrija Starpi\\nstarp to Uhdeni/ kas Isplattij\\nuna appakschā bija / un starp\\

In [8]:
import time
t0 = time.time()
r = extract_page(13)
print(f"{time.time()-t0:.1f}s  max={r.max_verse}  conf={r.confidence}")
print("raw response:", r.raw[:500])

NameError: name 'extract_page' is not defined

## 8. Batched inference (concurrent)

vLLM's server handles request-level batching internally — we just fire concurrent HTTP requests and it packs them. Thread pool works well here because each request is GPU-bound on the server side; we're just doing I/O.

In [16]:
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm.auto import tqdm


def extract_range(pages: list[int], batch: int = BATCH,
                   show_progress: bool = True) -> dict[int, PageResult]:
    results: dict[int, PageResult] = {}
    pbar = tqdm(total=len(pages), desc='pages') if show_progress else None
    with ThreadPoolExecutor(max_workers=batch) as ex:
        futures = {ex.submit(extract_page, p): p for p in pages}
        for f in as_completed(futures):
            p = futures[f]
            results[p] = f.result()
            if pbar:
                pbar.update(1)
                pbar.set_postfix(last=f'p{p}={results[p].max_verse}')
    if pbar:
        pbar.close()
    return results




In [ ]:
# Time a 20-page run to project total
import time
t0 = time.time()
sample = extract_range(list(range(13, 33)))
elapsed = time.time() - t0
print(f'\n20 pages in {elapsed:.1f}s  →  {elapsed/20:.2f}s/page')
print(f'Projected full book (~{LAST_PAGE - FIRST_PAGE + 1} pages): {(LAST_PAGE - FIRST_PAGE + 1) * elapsed / 20 / 60:.1f} min')

## 9. QA pass on known pages (before committing to full book)

Run everything we have ground truth for; spot-check the disagreement rate. If it's >10% something is wrong with the prompt — don't proceed to the full book yet.

In [33]:
gt_pages = sorted(PAGE_TO_EXPECTED.keys())
print(f'Running {len(gt_pages)} ground-truth pages...')
gt_results = extract_range(gt_pages)

# Build a richer expected set per page: all verses the ground truth says
# appear on that page (across chapters that span to/from it).
from collections import defaultdict

PAGE_EXPECTED = defaultdict(set)

for (_, _), (first, verses) in GROUND_TRUTH.items():
    p = first
    for v in verses:
        if isinstance(v, (int, float)) and v < 0:
            # Skip N pages forward; don't emit an expectation for any of them
            p += int(-v)
            continue
        exp = gt_int(v)
        if exp is not None:
            PAGE_EXPECTED[p].add(exp)
        p += 1

qa_pages = sorted(PAGE_EXPECTED.keys())
print(f'{len(qa_pages)} ground-truth pages loaded')

hits = misses = errors = 0
mismatches = []
for p in gt_pages:
    r = gt_results[p]
    expected = PAGE_EXPECTED[p]
    if r.error:
        errors += 1
        continue
    # Fine if max matches any expected, OR the model saw all expected numbers
    if r.max_verse in expected or expected.issubset(set(r.full_list)):
        hits += 1
    else:
        misses += 1
        mismatches.append((p, expected, r))

print(f'\nAccuracy: {hits}/{len(gt_pages)} = {hits/len(gt_pages)*100:.1f}%  ({misses} misses, {errors} errors)')
if mismatches:
    print('\nMismatches:')
    for p, exp, r in mismatches:#[:20]:
        alt_note = f' alts={r.alternatives}' if r.alternatives else ''
        print(f'  p{p}: got {r.max_verse} expected {exp}  conf={r.confidence}{alt_note}')

Running 113 ground-truth pages...


pages:   0%|          | 0/113 [00:00<?, ?it/s]

112 ground-truth pages loaded

Accuracy: 99/113 = 87.6%  (14 misses, 0 errors)

Mismatches:
  p56: got 38 expected {39}  conf=high
  p57: got 51 expected {52}  conf=high
  p63: got 33 expected {8, 35}  conf=high
  p117: got 17 expected {18}  conf=high
  p127: got 12 expected {13, 55}  conf=high alts=[[12, [13]]]
  p135: got 22 expected {24}  conf=high
  p137: got 13 expected {14}  conf=high
  p141: got 29 expected {5, 30}  conf=high
  p143: got 21 expected {11, 23}  conf=high
  p146: got 34 expected {35}  conf=high
  p149: got 22 expected {23}  conf=high
  p156: got 11 expected {34, 12}  conf=high
  p168: got 26 expected {9}  conf=high
  p176: got 14 expected {15}  conf=high


In [31]:
(RESULTS / f'00061.json').unlink()
r = extract_page(61)
print(r.full_list)
print(r.raw[:600])

[34, 6, 7, 8, 9, 10, 11]
{
  "verses": [
    {"n": 34, "column": "left", "alternatives": []},
    {"n": 6, "column": "right", "alternatives": []},
    {"n": 7, "column": "right", "alternatives": []},
    {"n": 8, "column": "right", "alternatives": []},
    {"n": 9, "column": "right", "alternatives": []},
    {"n": 10, "column": "right", "alternatives": []},
    {"n": 11, "column": "right", "alternatives": []}
  ],
  "chapter_starts_here": true,
  "confidence": "high",
  "notes": ""
}


In [32]:
# Clear cache for all remaining pattern-B mismatches
pattern_b_pages = [63, 67, 169, 177]
for p in pattern_b_pages:
    (RESULTS / f'{p:05d}.json').unlink(missing_ok=True)

retry = extract_range(pattern_b_pages)
for p in pattern_b_pages:
    r = retry[p]
    expected = PAGE_EXPECTED.get(p, set())
    status = "OK" if r.max_verse in expected else "MISS"
    print(f'p{p}: {status}  got={r.max_verse}  expected={expected}  list={r.full_list}')

pages:   0%|          | 0/4 [00:00<?, ?it/s]

p63: MISS  got=33  expected={8, 35}  list=[29, 30, 31, 32, 33, 1, 2, 3, 4, 5, 6, 7, 8]
p67: MISS  got=27  expected={46}  list=[10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27]
p169: MISS  got=22  expected={1, 11}  list=[1, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22]
p177: OK  got=23  expected={23}  list=[15, 16, 7, 8, 9, 10, 11, 12, 13, 14, 15, 23, 1]


## 10. Full-book pass

Only run this if QA above looks good.

Per-page results are cached to `~/gluck_vlm/results/NNNNN.json`, so if the notebook dies you can just re-run this cell — already-extracted pages are re-read from disk. That's what makes this safe to leave unattended.

In [17]:
all_pages = list(range(FIRST_PAGE, LAST_PAGE + 1))
print(f'extracting {len(all_pages)} pages...')
all_results = extract_range(all_pages, batch=BATCH)

n_err = sum(1 for r in all_results.values() if r.error)
n_low = sum(1 for r in all_results.values() if r.confidence == 'low')
n_med = sum(1 for r in all_results.values() if r.confidence == 'medium')
n_none = sum(1 for r in all_results.values() if r.max_verse is None and not r.error)
print(f'\nDone. {len(all_results)} pages extracted.')
print(f'  {n_err} errors')
print(f'  {n_low} low-confidence')
print(f'  {n_med} medium-confidence')
print(f'  {n_none} pages with no body verses detected')

extracting 2678 pages...


pages:   0%|          | 0/2678 [00:00<?, ?it/s]


Done. 2678 pages extracted.
  2022 errors
  2 low-confidence
  0 medium-confidence
  2 pages with no body verses detected


In [18]:
n_none

2

## 11. Re-check uncertain pages with higher-resolution crop

For low/medium-confidence pages, redo with a larger image. Sometimes a medium-confidence read on a shrunk image becomes a confident one when given more pixels.

In [11]:
def recheck_page(page: int) -> PageResult:
    """Same as extract_page but with doubled pixel budget, no crop."""
    try:
        img = fetch_image(page)
        # No body crop this time — model sometimes uses margin context to disambiguate.
        # Higher pixel cap.
        img = shrink_for_vlm(img, max_pixels=2 * IMG_MAX_PIXELS)
        resp = _client.chat.completions.create(
            model=MODEL, temperature=0.0, max_tokens=1024,
            messages=[{
                'role': 'user',
                'content': [
                    {'type': 'image_url', 'image_url': {'url': img_to_data_url(img)}},
                    {'type': 'text', 'text': PROMPT},
                ],
            }],
            extra_body={"chat_template_kwargs": {"enable_thinking": False}},
        )
        return parse(page, resp.choices[0].message.content)
    except Exception as e:
        return PageResult(page=page, error=str(e))


uncertain = sorted([
    p for p, r in all_results.items()
    if r.error or r.confidence in ('low', 'medium')
    or (r.max_verse is None and not any(s in r.notes.lower() for s in ['blank', 'title', 'no body']))
])
print(f'{len(uncertain)} uncertain pages. Re-checking...')

with ThreadPoolExecutor(max_workers=BATCH) as ex:
    rechecked = dict(zip(uncertain, tqdm(ex.map(recheck_page, uncertain), total=len(uncertain))))

# Merge: rechecked overrides all_results where the new confidence is higher
CONF_RANK = {'low': 0, 'medium': 1, 'high': 2, 'unknown': -1}
for p, r2 in rechecked.items():
    r1 = all_results[p]
    if r2.error:
        continue
    if CONF_RANK.get(r2.confidence, -1) > CONF_RANK.get(r1.confidence, -1):
        all_results[p] = r2
        # Rewrite cache
        (RESULTS / f'{p:05d}.json').write_text(json.dumps(r2.__dict__))

still_uncertain = [p for p, r in all_results.items()
                   if r.error or r.confidence in ('low',)]
print(f'\n{len(still_uncertain)} pages still uncertain after re-check — these need manual QA')

NameError: name 'all_results' is not defined

## 12. Review still-uncertain pages

Dump a compact table so you can eyeball which pages need human review.

In [ ]:
print(f'{"page":>6} | {"max":>4} | {"conf":>6} | {"chapter?":>8} | notes')
print('-' * 80)
for p in sorted(all_results.keys()):
    r = all_results[p]
    if r.confidence != 'low' and not r.error and r.max_verse is not None:
        continue
    note = (r.error or r.notes)[:40]
    cs = 'yes' if r.chapter_starts_here else '-'
    print(f'{p:>6} | {str(r.max_verse):>4} | {r.confidence:>6} | {cs:>8} | {note}')

## 13. Group into chapters and format as your mapping dict

Using the `chapter_starts_here` signal from each page, group consecutive pages into chapters, then emit the tuple format you already use.

In [ ]:
# Group pages into chapter runs based on chapter_starts_here flag.
# This produces a list of (first_page, [max_verse, ...]) for each chapter the
# model identified. Book name / chapter number you'd still assign manually,
# but the structure of the book is deterministic so it's easy to walk through.

def group_chapters(results: dict[int, PageResult]) -> list[tuple[int, list]]:
    chapters = []
    current_first = None
    current_maxes: list = []
    for p in sorted(results.keys()):
        r = results[p]
        if r.chapter_starts_here:
            if current_first is not None:
                chapters.append((current_first, current_maxes))
            current_first = p
            current_maxes = [r.max_verse]
        elif current_first is not None:
            current_maxes.append(r.max_verse)
    if current_first is not None:
        chapters.append((current_first, current_maxes))
    return chapters


chapters = group_chapters(all_results)
print(f'detected {len(chapters)} chapter runs')
print('\nFirst 20:')
for first, maxes in chapters[:20]:
    print(f'  ({first}, {maxes})')

## 14. Write the final mapping file

Dumps a Python file with the raw per-page results (so you can plug book names in manually) and a best-effort mapping dict template.

In [ ]:
from datetime import datetime

out_file = WORK / 'bible_mapping.py'
lines = [
    f'# Gluck 1694 Latvian Fraktur Bible — verse number mapping',
    f'# Auto-generated {datetime.now().isoformat()}',
    f'# Model: {MODEL}',
    f'# Mirror: {MIRROR_BASE}',
    f'',
    f'# Per-page: (max_verse, confidence, alternatives, notes)',
    f'PAGES = {{',
]
for p in sorted(all_results.keys()):
    r = all_results[p]
    lines.append(f'    {p}: ({r.max_verse!r}, {r.confidence!r}, {r.alternatives!r}, {r.notes!r}),')
lines.append('}')
lines.append('')
lines.append('# Detected chapter runs, ready to assign (book, chapter) keys')
lines.append('CHAPTERS = [')
for first, maxes in chapters:
    lines.append(f'    ({first}, {maxes!r}),')
lines.append(']')

out_file.write_text('\n'.join(lines))
print(f'wrote {out_file}  ({out_file.stat().st_size:,} bytes)')
print(f'\nFirst 40 lines:')
print('\n'.join(lines[:40]))